# A/B-тест лендинга: влияет ли новая версия на конверсию?

**Учебный проект по продуктовой аналитике.** Исходное задание переработано в самостоятельный аналитический кейс; коммерческий заказчик и точный бизнес-бриф неизвестны.

**Цель:** проверить, есть ли статистически обоснованные свидетельства изменения конверсии при переходе с `old_page` на `new_page`.

**Данные:** `user_id`, `timestamp`, `group`, `landing_page`, `converted` (0/1). Исходный CSV не публикуется из-за неподтверждённого права на распространение.

## 1. Подготовка окружения и загрузка данных

Размещаем оригинальный `data_ab.csv` рядом с Notebook. Данные не генерируются искусственно, чтобы не подменять результаты эксперимента.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEED = 42
path = Path("data_ab.csv")
if not path.exists():
    raise FileNotFoundError("Для запуска положите оригинальный data_ab.csv рядом с Notebook.")
df = pd.read_csv(path, parse_dates=["timestamp"])
required = {"user_id", "timestamp", "group", "landing_page", "converted"}
assert required.issubset(df.columns), f"Нет столбцов: {required - set(df.columns)}"
print("Строк в исходных данных:", len(df))
display(df.head())
print(df.isna().sum())

## 2. Контроль качества эксперимента

Проверяем соответствие назначенной группы и фактически показанной страницы. Исключаем несогласованные строки и оставляем первое согласованное наблюдение каждого пользователя. Это аналитическое допущение: в реальном эксперименте причины несовпадений нужно изучить отдельно.

In [ ]:
display(pd.crosstab(df["group"], df["landing_page"]))
valid = ((df["group"] == "control") & (df["landing_page"] == "old_page")) | ((df["group"] == "treatment") & (df["landing_page"] == "new_page"))
clean = (df.loc[valid].sort_values(["user_id","timestamp"]).drop_duplicates("user_id", keep="first").copy())
assert clean["user_id"].is_unique
assert clean["converted"].isin([0,1]).all()
print("Уникальных пользователей после очистки:",len(clean))
print("Период:",clean["timestamp"].min(),clean["timestamp"].max())

### SRM: проверка соотношения групп

Если плановый сплит был 50/50, тест χ² позволяет обнаружить крупное отклонение от ожидаемых размеров групп. **50/50 — допущение**, так как первоначальный дизайн эксперимента не приложен.

In [ ]:
counts = clean["group"].value_counts().reindex(["control","treatment"])
chi2, srm_p = stats.chisquare(counts, f_exp=[len(clean)/2]*2)
print("Участников:", counts.to_dict())
print(f"SRM: χ²={chi2:.5f}, p={srm_p:.4f}")

## 3. Конверсия и её динамика

Метрика: доля пользователей с `converted=1`. Сравним средние конверсии в группах и посмотрим изменение числа пользователей и конверсии по дням.

In [ ]:
conv = clean.groupby("group")["converted"].agg(["sum","count","mean"]).reindex(["control","treatment"])
display(conv.rename(columns={"sum":"Конверсии","count":"Пользователи","mean":"CR"}))
p_control,p_test = conv.loc["control","mean"],conv.loc["treatment","mean"]
diff=p_test-p_control
print(f"CR контроль: {p_control:.3%}; тест: {p_test:.3%}; разница: {diff*100:.3f} п.п.")
ax=(conv["mean"]*100).plot(kind="bar",ylabel="Конверсия, %",title="Конверсия по группам",rot=0)
plt.show()

In [ ]:
daily=(clean.assign(date=clean["timestamp"].dt.date).groupby(["date","group"]).agg(users=("converted","size"),conversion=("converted","mean")).reset_index())
for metric, ylabel in [("users","Пользователи"),("conversion","Конверсия")]:
    fig,ax=plt.subplots(figsize=(10,3))
    for group,g in daily.groupby("group"):
        ax.plot(g["date"],g[metric],label=group)
    ax.set(title=f"По дням: {ylabel}",ylabel=ylabel)
    ax.legend();fig.autofmt_xdate();plt.show()

## 4. Статистическая проверка

**H₀:** конверсии равны. **H₁:** конверсии различаются. Двусторонний z-тест двух долей с объединённой оценкой дисперсии; для интервала эффекта используем необъединённую стандартную ошибку. Уровень значимости 5%.

In [ ]:
n0,n1=conv.loc["control","count"],conv.loc["treatment","count"]
x0,x1=conv.loc["control","sum"],conv.loc["treatment","sum"]
pooled=(x0+x1)/(n0+n1)
se_null=np.sqrt(pooled*(1-pooled)*(1/n0+1/n1))
z=diff/se_null
p_value=2*stats.norm.sf(abs(z))
se_ci=np.sqrt(p_control*(1-p_control)/n0+p_test*(1-p_test)/n1)
zcrit=stats.norm.ppf(.975)
ci=(diff-zcrit*se_ci,diff+zcrit*se_ci)
print(f"z={z:.4f}; двустороннее p-value={p_value:.4f}")
print(f"Разница: {diff*100:.3f} п.п.; 95% ДИ: [{ci[0]*100:.3f}; {ci[1]*100:.3f}] п.п.")

## 5. Bootstrap-оценка эффекта

Повторно выбираем наблюдения с возвращением **внутри каждой группы** и рассчитываем разницу конверсий. Получим 95% процентильный интервал.

In [ ]:
rng=np.random.default_rng(SEED)
a=clean.loc[clean["group"]=="control","converted"].to_numpy()
b=clean.loc[clean["group"]=="treatment","converted"].to_numpy()
# Чанки вместо одной большой матрицы индексов, чтобы сократить потребление RAM.
boot=[]
for _ in range(3000):
    boot.append(b[rng.integers(0,len(b),len(b))].mean()-a[rng.integers(0,len(a),len(a))].mean())
boot=np.array(boot)
boot_ci=np.quantile(boot,[.025,.975])
print(f"Bootstrap 95% ДИ: [{boot_ci[0]*100:.3f}; {boot_ci[1]*100:.3f}] п.п.")
plt.figure(figsize=(8,3));plt.hist(boot*100,bins=40);plt.axvline(0,color="black",linestyle="--");plt.xlabel("Разница конверсий, п.п.");plt.show()

## 6. Продуктовый вывод

При повторном расчёте по приложенному исходному набору (не опубликованному в репозитории) получены: **290 584** пользователей после очистки, **12,039%** в контроле и **11,881%** в тесте, разница **−0,158 п.п.**, двустороннее **p ≈ 0,190**. Доверительный интервал включает ноль.

**Рекомендация:** нет статистически достаточных оснований внедрять новый лендинг исключительно ради повышения конверсии. Это не означает доказанной эквивалентности вариантов.

**Ограничения:** бизнес-бриф, план рандомизации и подтверждённое распределение трафика не представлены; исходный CSV нельзя автоматически считать открытым. SRM не заменяет проверку корректности рандомизации, а очистка может влиять на оценку эффекта.